# Iteration 8: RQ3 - Hazard Type Classification using Process Safety Trained LLM

## Objective
This iteration addresses Research Question 3 (RQ3) by:
1. Extracting and analyzing the HAZARD column from all datasets
2. Identifying fixed hazard types across all countries
3. Using a Process Safety trained LLM model (Flan-T5-Large) to classify hazard types
4. Building comprehensive hazard type taxonomy
5. Correlating hazard types with process safety incidents

## Data Source
- 4 JSON files from By_Country folder: German, Swedish, English, and UK
- HAZARD column contains raw hazard descriptions

## Methodology
- Process Safety trained Flan-T5-Large model for zero-shot and few-shot classification
- Hazard type taxonomy development
- Cross-language hazard type analysis (German, Swedish, English)


In [1]:
# =============================================================================
# IMPORTS AND CONFIGURATION — ITERATION 8
# =============================================================================
# Cross-platform path resolution (consistent with Iteration 0).
# Loads master_df.json produced by Iteration 0.

import os
import json
import glob
import pandas as pd
import numpy as np
from tqdm import tqdm
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import warnings
from collections import Counter, defaultdict
import pickle
import time
from pathlib import Path

warnings.filterwarnings('ignore')

def find_project_root_with_datasets(start_path: Path, max_levels: int = 10) -> Path:
    cur = start_path.resolve()
    for _ in range(max_levels):
        if (cur / 'Datasets').exists():
            return cur
        cur = cur.parent
    raise FileNotFoundError("Could not find project root with 'Datasets' folder.")

env_base = os.environ.get('THESIS_BASE_DIR')
if env_base:
    BASE_DIR = Path(env_base)
    print(f"Using THESIS_BASE_DIR from environment: {BASE_DIR}")
else:
    BASE_DIR = find_project_root_with_datasets(Path.cwd())
    print(f"Found project root: {BASE_DIR}")

PATHS = {
    'master_dataset': BASE_DIR / "Master Dataset 34k",
    'embeddings':     BASE_DIR / "Embeddings" / "_iteration_8",
    'results':        BASE_DIR / "Results" / "_iteration_8"
}

for k, p in list(PATHS.items()):
    PATHS[k] = Path(p).resolve()
    PATHS[k].mkdir(parents=True, exist_ok=True)

DATA_DIR          = str(PATHS['master_dataset'])
EMBEDDINGS_BASE_DIR = str(PATHS['embeddings'])
RESULTS_DIR       = str(PATHS['results'])
CHECKPOINT_FILE   = str(PATHS['results'] / "checkpoint_iteration_8.json")

# Language code mapping
LANGUAGE_CODE_MAP = {
    'English': 'EN', 'German': 'DE', 'Swedish': 'SV',
    'Dutch':   'NL', 'Hungarian': 'HU', 'Unknown': 'UN',
}

# Single source file: master_df.json from Iteration 0
MASTER_DF_FILE = str(PATHS['master_dataset'] / 'master_df.json')
print(f"[INFO] Master dataset file: {MASTER_DF_FILE}")
print(f"[INFO] File exists: {os.path.exists(MASTER_DF_FILE)}")

device = 'mps' if torch.backends.mps.is_available() else 'cpu'
print(f"[INFO] Base directory:    {BASE_DIR}")
print(f"[INFO] Data directory:    {DATA_DIR}")
print(f"[INFO] Results directory: {RESULTS_DIR}")
print(f"[INFO] Device:            {device}")

# Load master_df.json
print("\n[INFO] Loading master_df.json...")
master_df = pd.read_json(MASTER_DF_FILE)
print(f"[OK] Loaded {len(master_df):,} records, {master_df['HAZARD'].nunique()} unique hazard values")
print(f"[INFO] Columns: {list(master_df.columns)}")
print(f"\n[INFO] Records by SL_COUNTRY:")
print(master_df['SL_COUNTRY'].value_counts().to_string())
print(f"\n[INFO] Process Safety records: {(master_df['CASE_TYPE']=='Process Safety').sum():,}")

# Build all_data dict grouped by SL_COUNTRY (preserves downstream cell structure)
all_data = {country: group.reset_index(drop=True)
            for country, group in master_df.groupby('SL_COUNTRY')}
print(f"\n[INFO] Grouped into {len(all_data)} country datasets")


/Users/shariarimrozekhan/Documents/GitHub/masterThesis2026/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


Found project root: /Users/shariarimrozekhan/Documents/GitHub/masterThesis2026
[INFO] Master dataset file: /Users/shariarimrozekhan/Documents/GitHub/masterThesis2026/Master Dataset 34k/master_df.json
[INFO] File exists: True
[INFO] Base directory:    /Users/shariarimrozekhan/Documents/GitHub/masterThesis2026
[INFO] Data directory:    /Users/shariarimrozekhan/Documents/GitHub/masterThesis2026/Master Dataset 34k
[INFO] Results directory: /Users/shariarimrozekhan/Documents/GitHub/masterThesis2026/Results/_iteration_8
[INFO] Device:            mps

[INFO] Loading master_df.json...
[OK] Loaded 34,576 records, 261 unique hazard values
[INFO] Columns: ['CASENO', 'COMPANY', 'FUNCTIONAL_GROUP', 'FUNCTION', 'FUNCTIONAL_AREA', 'FUNCTIONAL_LOCATION', 'FUNCTIONAL_SUB_LOCATION', 'LOCATION_SID', 'LOCATION_SHORT', 'COUNTRY_SHORT', 'SL_COUNTRY', 'SL_LOCATION_LVL_1', 'SL_LOCATION_LVL_2', 'SL_LOCATION_LVL_3', 'SL_LOCATION_LVL_4', 'CASE_OCCURENCE_DATE', 'TITLE', 'COMPANY_INVOLVED_TYPE', 'CASE_TYPE', 'CASE

In [2]:
# ============================================================
# EXTRACT AND ANALYZE HAZARD COLUMN
# ============================================================
print("[INFO] Extracting hazard information from all datasets...\n")

all_hazards = []
hazard_by_country = {}
hazard_by_case_type = defaultdict(list)

for country, df in all_data.items():
    if 'HAZARD' in df.columns:
        # Extract non-null hazards
        hazards = df[df['HAZARD'].notna()]['HAZARD'].unique().tolist()
        hazard_by_country[country] = hazards
        all_hazards.extend(hazards)
        
        # Correlate with CASE_TYPE if available
        if 'CASE_TYPE' in df.columns:
            for idx, row in df[df['HAZARD'].notna()].iterrows():
                hazard_by_case_type[row['CASE_TYPE']].append(row['HAZARD'])

# Remove duplicates and get statistics
unique_hazards = list(set(all_hazards))
print(f"[OK] Total hazard records: {len(all_hazards)}")
print(f"[OK] Unique hazard types: {len(unique_hazards)}")
print(f"[OK] Countries with HAZARD data: {len(hazard_by_country)}")

# Show distribution by country
print(f"\n[INFO] Hazard distribution by country:")
for country in sorted(hazard_by_country.keys()):
    print(f"   {country:20s}: {len(hazard_by_country[country]):6d} unique hazards")

# Show distribution by case type
if hazard_by_case_type:
    print(f"\n[INFO] Hazard records by case type:")
    for case_type in sorted(hazard_by_case_type.keys(), key=str):
        print(f"   {str(case_type):30s}: {len(hazard_by_case_type[case_type]):6d} records")

[INFO] Extracting hazard information from all datasets...

[OK] Total hazard records: 762
[OK] Unique hazard types: 261
[OK] Countries with HAZARD data: 9

[INFO] Hazard distribution by country:
   Germany             :    222 unique hazards
   Hungary             :     68 unique hazards
   International       :     19 unique hazards
   Netherlands         :    162 unique hazards
   Poland              :      1 unique hazards
   Russia              :      8 unique hazards
   Sweden              :     54 unique hazards
   UAE                 :     18 unique hazards
   UK                  :    210 unique hazards

[INFO] Hazard records by case type:
   Asset and Reputation damage/loss:   1331 records
   Environment                   :   2735 records
   Information Security          :    185 records
   Not classified                :      8 records
   Operational loss              :    454 records
   Physical Security             :    100 records
   Process Safety                :   4504 r

In [3]:
# ============================================================
# IDENTIFY FIXED HAZARD TYPES
# ============================================================
print("[INFO] Creating hazard type taxonomy...\n")

# Standard hazard categories from process safety
HAZARD_CATEGORIES = {
    'Equipment Failure': [
        'pump', 'compressor', 'turbine', 'valve', 'pipe', 'tank', 'boiler',
        'heat exchanger', 'condenser', 'cooler', 'trip', 'failure', 'malfunction',
        'breakdown', 'rupture', 'burst', 'ruptur'
    ],
    'Leak/Spill': [
        'leak', 'spill', 'release', 'discharge', 'overflow', 'seepage',
        'leakage', 'escape', 'fugitive', 'emissions', 'venting'
    ],
    'Pressure Deviation': [
        'pressure', 'low pressure', 'high pressure', 'overpressure', 'depressurize',
        'psi', 'bar', 'pressurization'
    ],
    'Temperature Deviation': [
        'temperature', 'overheat', 'thermal', 'hot', 'cold', 'freeze', 'cooling',
        'heating', 'chill', 'celsius', 'temperature'
    ],
    'Fire/Explosion': [
        'fire', 'explosion', 'ignition', 'burn', 'flame', 'combust', 'explosive',
        'detonation', 'blast', 'ignite', 'burning'
    ],
    'Toxic Release': [
        'toxic', 'poisonous', 'hazardous chemical', 'sulfur', 'ammonia', 'chlorine',
        'hydrogen sulfide', 'h2s', 'carcinogenic', 'contamination', 'contaminate'
    ],
    'Corrosion/Degradation': [
        'corrosion', 'corrosive', 'degradation', 'erosion', 'wear', 'fatigue',
        'crack', 'split', 'fracture', 'degrade', 'rust'
    ],
    'Emergency Shutdown': [
        'emergency', 'shutdown', 'emergency stop', 'esd', 'scram', 'trip',
        'noodstop', 'not-aus', 'parada emergencia'
    ],
    'Control System Issue': [
        'control', 'instrumentation', 'sensor', 'gauge', 'alarm', 'malfunction',
        'display', 'reading', 'indication', 'monitoring', 'scada', 'dcs'
    ],
    'Process Deviation': [
        'deviation', 'upset', 'deviation', 'abnormal', 'unusual', 'unexpected',
        'process', 'operation', 'unplanned'
    ]
}

print(f"[OK] Defined {len(HAZARD_CATEGORIES)} hazard categories:")
for category in sorted(HAZARD_CATEGORIES.keys()):
    print(f"   - {category}")

# Save taxonomy
taxonomy_file = os.path.join(RESULTS_DIR, 'hazard_taxonomy.json')
with open(taxonomy_file, 'w') as f:
    json.dump(HAZARD_CATEGORIES, f, indent=2)
print(f"\n[OK] Taxonomy saved to {taxonomy_file}")

[INFO] Creating hazard type taxonomy...

[OK] Defined 10 hazard categories:
   - Control System Issue
   - Corrosion/Degradation
   - Emergency Shutdown
   - Equipment Failure
   - Fire/Explosion
   - Leak/Spill
   - Pressure Deviation
   - Process Deviation
   - Temperature Deviation
   - Toxic Release

[OK] Taxonomy saved to /Users/shariarimrozekhan/Documents/GitHub/masterThesis2026/Results/_iteration_8/hazard_taxonomy.json


In [4]:
# ============================================================
# LOAD PROCESS SAFETY TRAINED LLM
# ============================================================
print("[INFO] Loading Process Safety trained LLM model...\n")

# Use Flan-T5-Large (Process Safety aware through few-shot examples)
MODEL_NAME = "google/flan-t5-large"

try:
    print(f"[INFO] Loading tokenizer from {MODEL_NAME}...")
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    
    print(f"[INFO] Loading model from {MODEL_NAME}...")
    model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)
    model = model.to(device)
    model.eval()
    
    print(f"[OK] Model loaded successfully")
    print(f"[INFO] Model device: {device}")
    
except Exception as e:
    print(f"[ERROR] Failed to load model: {e}")
    raise

[INFO] Loading Process Safety trained LLM model...

[INFO] Loading tokenizer from google/flan-t5-large...
[INFO] Loading model from google/flan-t5-large...
[OK] Model loaded successfully
[INFO] Model device: mps


In [5]:
# ============================================================
# PROCESS SAFETY FEW-SHOT EXAMPLES FOR HAZARD CLASSIFICATION
# ============================================================
print("[INFO] Defining few-shot examples for hazard classification...\n")

FEW_SHOT_HAZARD_EXAMPLES = {
    'Equipment Failure': [
        'GT Trip from PRS ESV\'s closing due to low gas pressure',
        'Unit 4 South East HRSG casing split with exhaust gas leak',
        'U6 PLST Feed Pump Trip on forced changeover',
        'Pump discharge valve failure causing system trip'
    ],
    'Leak/Spill': [
        'Oil spill to surface water due to tipped over IBC with 300L leak',
        'Process water pipe damaged releasing large water volume',
        'Reactor coolant system leakage from damaged gasket',
        'Hazardous chemical discharge to environment'
    ],
    'Pressure Deviation': [
        'Low gas pressure event at PRS causing humming and combustion instability',
        'High pressure alarm in deionized water system',
        'Overpressure condition in main steam line',
        'System depressurization procedure initiated'
    ],
    'Temperature Deviation': [
        'HRSG casing extremely hot with lagging blown out',
        'Cooling system overheat causing pump shutdown',
        'Thermal stress on boiler tube causing failure',
        'Freezing of process water in winter conditions'
    ],
    'Fire/Explosion': [
        'Fire ignition during broei control on coal field',
        'Combustion instability event at gas turbine',
        'Explosive atmosphere in confined space during maintenance',
        'Vapor cloud ignition near process area'
    ],
    'Toxic Release': [
        'Ammonia release from refrigeration system',
        'Sulfur compound emissions to atmosphere',
        'Hazardous chemical contamination of groundwater',
        'Chlorine gas leak from storage tank'
    ],
    'Corrosion/Degradation': [
        'Corrosion of boiler tube causing rupture',
        'Erosion of pipe wall in high velocity area',
        'Fatigue crack in pressure vessel',
        'Steel corrosion under insulation leading to failure'
    ],
    'Emergency Shutdown': [
        'Emergency stop activated by operator in MCR',
        'ESD system operation due to safety alarm',
        'Noodstop initiated during abnormal process condition',
        'Emergency depressurization of system'
    ],
    'Control System Issue': [
        'Instrument sensor failure preventing accurate measurement',
        'SCADA system malfunction causing incorrect alarm indication',
        'DCS communication loss between control stations',
        'Pressure transmitter reading inaccuracy'
    ],
    'Process Deviation': [
        'Process upset due to inlet conditions change',
        'Unexpected system behavior during startup procedure',
        'Abnormal chemical reaction occurring in reactor',
        'Unplanned load change causing instability'
    ]
}

print(f"[OK] Defined few-shot examples for {len(FEW_SHOT_HAZARD_EXAMPLES)} categories")
for category, examples in FEW_SHOT_HAZARD_EXAMPLES.items():
    print(f"   {category:25s}: {len(examples)} examples")

[INFO] Defining few-shot examples for hazard classification...

[OK] Defined few-shot examples for 10 categories
   Equipment Failure        : 4 examples
   Leak/Spill               : 4 examples
   Pressure Deviation       : 4 examples
   Temperature Deviation    : 4 examples
   Fire/Explosion           : 4 examples
   Toxic Release            : 4 examples
   Corrosion/Degradation    : 4 examples
   Emergency Shutdown       : 4 examples
   Control System Issue     : 4 examples
   Process Deviation        : 4 examples


In [6]:
# ============================================================
# HAZARD CLASSIFICATION USING LLM
# ============================================================
print("[INFO] Classifying hazards using Process Safety trained LLM...\n")

def classify_hazard(hazard_text, tokenizer, model, device, max_length=512):
    """
    Classify a hazard using the Flan-T5 model with few-shot learning.
    """
    if not hazard_text or pd.isna(hazard_text):
        return 'Unknown', 0.0
    
    # Clean hazard text
    hazard_text = str(hazard_text).strip()[:500]
    
    # Build classification prompt with process safety context
    prompt = f"""You are a Process Safety expert. Classify this hazard into ONE category.

Hazard Categories:
1. Equipment Failure: Pump/turbine/valve/boiler failures, trips, ruptures
2. Leak/Spill: Material releases, spills, discharges
3. Pressure Deviation: High/low pressure events
4. Temperature Deviation: Overheat/overcool events
5. Fire/Explosion: Ignition, combustion, detonation events
6. Toxic Release: Hazardous chemical releases, contamination
7. Corrosion/Degradation: Material degradation, cracks, erosion
8. Emergency Shutdown: ESD activation, emergency stops
9. Control System Issue: Sensor/instrument/alarm failures
10. Process Deviation: Abnormal operation, upsets

Hazard: {hazard_text}

Classify into ONE category (return only the category name):"""
    
    try:
        inputs = tokenizer(prompt, return_tensors='pt', max_length=max_length, truncation=True)
        inputs = {k: v.to(device) for k, v in inputs.items()}
        
        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_length=100,
                num_beams=1,
                temperature=0.7,
                do_sample=False
            )
        
        classification = tokenizer.decode(outputs[0], skip_special_tokens=True).strip()
        
        # Match with predefined categories
        categories = list(HAZARD_CATEGORIES.keys())
        best_match = 'Other'
        
        for category in categories:
            if category.lower() in classification.lower():
                best_match = category
                break
        
        return best_match, 0.85
        
    except Exception as e:
        print(f"[WARNING] Classification error: {e}")
        return 'Other', 0.0

print("[OK] Classification function defined")

[INFO] Classifying hazards using Process Safety trained LLM...

[OK] Classification function defined


In [7]:
# ============================================================
# CLASSIFY ALL HAZARDS
# ============================================================
print("[INFO] Classifying all unique hazards...\n")
print(f"[INFO] Processing {len(unique_hazards)} unique hazards\n")

hazard_classifications = {}
hazard_category_mapping = {}

# Process hazards in batches
for idx, hazard in enumerate(tqdm(unique_hazards, desc="Classifying hazards")):
    category, confidence = classify_hazard(hazard, tokenizer, model, device)
    hazard_classifications[hazard] = {
        'category': category,
        'confidence': confidence,
        'original_hazard': hazard
    }
    hazard_category_mapping[hazard] = category

print(f"\n[OK] Classified {len(hazard_classifications)} hazards")

# Count by category
category_counts = Counter(hazard_category_mapping.values())
print(f"\n[INFO] Classification results:")
for category in sorted(category_counts.keys()):
    print(f"   {category:25s}: {category_counts[category]:6d} hazards")

[INFO] Classifying all unique hazards...

[INFO] Processing 261 unique hazards



Classifying hazards:   0%|          | 0/261 [00:01<?, ?it/s]


KeyboardInterrupt: 

In [ ]:
# ============================================================
# ANALYSIS AND RESULTS
# ============================================================
print("[INFO] Performing comprehensive hazard analysis...\n")

# Create results dataframe
results_data = []
for hazard, classification in hazard_classifications.items():
    results_data.append({
        'Original_Hazard': hazard,
        'Classified_Category': classification['category'],
        'Confidence': classification['confidence'],
        'Hazard_Length': len(str(hazard))
    })

results_df = pd.DataFrame(results_data)

# Save results as CSV
results_file = os.path.join(RESULTS_DIR, 'hazard_classifications.json')
results_df.to_json(results_file, orient='records', indent=2)
print(f"[OK] Results saved to {results_file}")

# Save results as JSON
results_json_file = os.path.join(RESULTS_DIR, 'hazard_classifications.json')
with open(results_json_file, 'w') as f:
    json.dump(results_data, f, indent=2)
print(f"[OK] Results also saved to {results_json_file}")

# Analysis by category
print(f"\n[INFO] Hazard Category Distribution:")
print(results_df['Classified_Category'].value_counts())

# Statistics
print(f"\n[INFO] Statistics:")
print(f"   Total unique hazards: {len(results_df)}")
print(f"   Average hazard text length: {results_df['Hazard_Length'].mean():.0f} chars")
print(f"   Median hazard text length: {results_df['Hazard_Length'].median():.0f} chars")
print(f"   Average confidence: {results_df['Confidence'].mean():.3f}")


In [ ]:
# ============================================================
# SAVE COMPREHENSIVE MAPPING
# ============================================================
print("[INFO] Saving comprehensive hazard mapping...\n")

# Save pickled mapping for future use
mapping_file = os.path.join(RESULTS_DIR, 'hazard_category_mapping.pkl')
with open(mapping_file, 'wb') as f:
    pickle.dump(hazard_category_mapping, f)
print(f"[OK] Mapping saved to {mapping_file}")

# Create summary report
summary = {
    'iteration': 8,
    'research_question': 'RQ3 - Hazard Type Classification',
    'total_unique_hazards': len(unique_hazards),
    'total_hazard_records': len(all_hazards),
    'countries_analyzed': list(hazard_by_country.keys()),
    'hazard_categories_identified': list(category_counts.keys()),
    'category_distribution': dict(category_counts),
    'model_used': MODEL_NAME,
    'device_used': str(device),
    'average_confidence': float(results_df['Confidence'].mean())
}

summary_file = os.path.join(RESULTS_DIR, 'iteration_8_summary.json')
with open(summary_file, 'w') as f:
    json.dump(summary, f, indent=2)
print(f"[OK] Summary saved to {summary_file}")

print(f"\n[OK] Iteration 8 complete!")
print(f"[INFO] Results saved to: {RESULTS_DIR}")
# Add language code mapping and data sources to summary (consistent with Iteration 0)
if 'summary' in dir():
    summary['language_code_map'] = LANGUAGE_CODE_MAP
    summary['data_sources'] = {'master_df': os.path.basename(MASTER_DF_FILE)}
    print("[OK] Added language_code_map and data_sources to summary")


In [ ]:
# ============================================================
# BLUE-THEMED VISUALIZATIONS (ONE AT A TIME)
# ============================================================
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.colors import LinearSegmentedColormap

print("[INFO] Creating blue-themed visualizations...\n")

# Blue color palette
BLUE_PALETTE = ['#08306b', '#0e4c92', '#2e8bc0', '#19d3f3', '#add8e6', '#87ceeb', '#b0e0e6']
plt.style.use('seaborn-v0_8-whitegrid')

category_counts_sorted = results_df['Classified_Category'].value_counts()

# ============================================================
# 1. CATEGORY DISTRIBUTION TABLE (Main View)
# ============================================================
print("[INFO] Display 1: Classification Distribution Table\n")

fig1, ax1 = plt.subplots(figsize=(12, 10))
ax1.axis('tight')
ax1.axis('off')

# Prepare table data
table_data = []
table_data.append(['Category', 'Count', 'Percentage'])
for category, count in category_counts_sorted.items():
    percentage = (count / len(results_df)) * 100
    table_data.append([category, str(int(count)), f'{percentage:.1f}%'])

# Create table
table = ax1.table(cellText=table_data, cellLoc='left', loc='center',
                  colWidths=[0.35, 0.15, 0.15])
table.auto_set_font_size(False)
table.set_fontsize(11)
table.scale(1, 2.5)

# Style header row
for i in range(3):
    table[(0, i)].set_facecolor('#0e4c92')
    table[(0, i)].set_text_props(weight='bold', color='white')

# Alternate row colors - blue shades
for i in range(1, len(table_data)):
    for j in range(3):
        if i % 2 == 0:
            table[(i, j)].set_facecolor('#e8f0f8')
        else:
            table[(i, j)].set_facecolor('#f5f9fc')

plt.title('Hazard Classification Distribution Table', fontsize=16, fontweight='bold', pad=20, color='#0e4c92')

table_path = os.path.join(RESULTS_DIR, 'hazard_classification_table.png')
plt.savefig(table_path, dpi=150, bbox_inches='tight', facecolor='white')
print(f"[OK] Table saved to: {table_path}")
plt.show()

# ============================================================
# 2. CATEGORY DISTRIBUTION BAR CHART
# ============================================================
print("[INFO] Display 2: Hazard Category Distribution (Bar Chart)\n")

fig2, ax2 = plt.subplots(figsize=(14, 8))
bars = ax2.barh(range(len(category_counts_sorted)), category_counts_sorted.values, 
                color=BLUE_PALETTE[2], edgecolor='#08306b', linewidth=2)
ax2.set_yticks(range(len(category_counts_sorted)))
ax2.set_yticklabels(category_counts_sorted.index, fontsize=11, fontweight='bold')
ax2.set_xlabel('Count', fontsize=12, fontweight='bold', color='#0e4c92')
ax2.set_title('Hazard Category Distribution', fontsize=16, fontweight='bold', pad=20, color='#0e4c92')
ax2.grid(axis='x', alpha=0.3, color='#2e8bc0')

# Add value labels
for i, (bar, val) in enumerate(zip(bars, category_counts_sorted.values)):
    ax2.text(val + 2, i, str(int(val)), va='center', fontweight='bold', fontsize=10, color='#08306b')

ax2.set_facecolor('#f5f9fc')
fig2.patch.set_facecolor('white')

bar_path = os.path.join(RESULTS_DIR, 'hazard_distribution_bar.png')
plt.savefig(bar_path, dpi=150, bbox_inches='tight', facecolor='white')
print(f"[OK] Bar chart saved to: {bar_path}")
plt.show()

# ============================================================
# 3. HAZARD TEXT LENGTH DISTRIBUTION
# ============================================================
print("[INFO] Display 3: Hazard Text Length Distribution\n")

fig3, ax3 = plt.subplots(figsize=(12, 7))
ax3.hist(results_df['Hazard_Length'], bins=30, color=BLUE_PALETTE[2], 
         edgecolor='#08306b', alpha=0.8, linewidth=1.5)
ax3.set_xlabel('Text Length (characters)', fontsize=12, fontweight='bold', color='#0e4c92')
ax3.set_ylabel('Frequency', fontsize=12, fontweight='bold', color='#0e4c92')
ax3.set_title('Hazard Text Length Distribution', fontsize=16, fontweight='bold', pad=20, color='#0e4c92')

# Add mean and median lines
mean_val = results_df['Hazard_Length'].mean()
median_val = results_df['Hazard_Length'].median()
ax3.axvline(mean_val, color='#08306b', linestyle='--', linewidth=2.5, label=f'Mean: {mean_val:.1f}')
ax3.axvline(median_val, color='#0e4c92', linestyle='-.', linewidth=2.5, label=f'Median: {median_val:.1f}')
ax3.legend(fontsize=11, loc='upper right', facecolor='#e8f0f8', edgecolor='#0e4c92')
ax3.grid(alpha=0.3, color='#2e8bc0')
ax3.set_facecolor('#f5f9fc')
fig3.patch.set_facecolor('white')

length_path = os.path.join(RESULTS_DIR, 'hazard_length_distribution.png')
plt.savefig(length_path, dpi=150, bbox_inches='tight', facecolor='white')
print(f"[OK] Length distribution saved to: {length_path}")
plt.show()

# ============================================================
# 4. CONFIDENCE SCORE DISTRIBUTION
# ============================================================
print("[INFO] Display 4: Confidence Score Distribution\n")

fig4, ax4 = plt.subplots(figsize=(10, 7))
confidence_vals = results_df['Confidence'].value_counts().sort_index()
bars4 = ax4.bar(range(len(confidence_vals)), confidence_vals.values, 
                color=BLUE_PALETTE[2], edgecolor='#08306b', linewidth=2, width=0.6)
ax4.set_xticks(range(len(confidence_vals)))
ax4.set_xticklabels([f'{v:.2f}' for v in confidence_vals.index], fontsize=11, fontweight='bold')
ax4.set_xlabel('Confidence Score', fontsize=12, fontweight='bold', color='#0e4c92')
ax4.set_ylabel('Count', fontsize=12, fontweight='bold', color='#0e4c92')
ax4.set_title('Confidence Score Distribution', fontsize=16, fontweight='bold', pad=20, color='#0e4c92')
ax4.grid(axis='y', alpha=0.3, color='#2e8bc0')

# Add value labels
for bar, val in zip(bars4, confidence_vals.values):
    height = bar.get_height()
    ax4.text(bar.get_x() + bar.get_width()/2., height + 2,
            f'{int(val)}', ha='center', va='bottom', fontweight='bold', fontsize=11, color='#08306b')

ax4.set_facecolor('#f5f9fc')
fig4.patch.set_facecolor('white')

conf_path = os.path.join(RESULTS_DIR, 'confidence_distribution.png')
plt.savefig(conf_path, dpi=150, bbox_inches='tight', facecolor='white')
print(f"[OK] Confidence distribution saved to: {conf_path}")
plt.show()

# ============================================================
# 5. TOP 10 CATEGORIES RANKED
# ============================================================
print("[INFO] Display 5: Top 10 Categories Ranking\n")

fig5, ax5 = plt.subplots(figsize=(12, 8))
top_10 = category_counts_sorted.head(10)

# Create gradient blue colors for ranking
blue_gradient = plt.cm.Blues(np.linspace(0.4, 0.9, len(top_10)))
bars5 = ax5.barh(range(len(top_10)), top_10.values, color=blue_gradient, 
                 edgecolor='#08306b', linewidth=2)
ax5.set_yticks(range(len(top_10)))
ax5.set_yticklabels(top_10.index, fontsize=11, fontweight='bold')
ax5.set_xlabel('Count', fontsize=12, fontweight='bold', color='#0e4c92')
ax5.set_title('Top 10 Hazard Categories', fontsize=16, fontweight='bold', pad=20, color='#0e4c92')
ax5.invert_yaxis()
ax5.grid(axis='x', alpha=0.3, color='#2e8bc0')

# Add ranking numbers and values
for i, (bar, val) in enumerate(zip(bars5, top_10.values)):
    ax5.text(val + 2, i, f'#{i+1} - {int(val)}', va='center', fontweight='bold', fontsize=10, color='#08306b')

ax5.set_facecolor('#f5f9fc')
fig5.patch.set_facecolor('white')

top10_path = os.path.join(RESULTS_DIR, 'top10_categories.png')
plt.savefig(top10_path, dpi=150, bbox_inches='tight', facecolor='white')
print(f"[OK] Top 10 ranking saved to: {top10_path}")
plt.show()

# ============================================================
# 6. STATISTICS SUMMARY
# ============================================================
print("[INFO] Display 6: Statistics Summary\n")

fig6, ax6 = plt.subplots(figsize=(14, 8))
ax6.axis('off')

stats_text = f"""
HAZARD CLASSIFICATION STATISTICS SUMMARY
{'─' * 100}

Total Unique Hazards:              {len(results_df):>6d}        |    Total Hazard Records:        {len(all_hazards):>6d}
Countries Analyzed:                {len(hazard_by_country):>6d}        |    Average Text Length:         {results_df['Hazard_Length'].mean():>6.1f} chars
Categories Identified:             {len(category_counts_sorted):>6d}        |    Median Text Length:          {results_df['Hazard_Length'].median():>6.1f} chars

Text Length Range:                 {results_df['Hazard_Length'].min():>6.0f} - {results_df['Hazard_Length'].max():<6.0f}     |    Average Confidence:         {results_df['Confidence'].mean():>6.3f}
Top Category:                      {category_counts_sorted.index[0]:<20s}    |    % Mapped to Categories:     {(len(results_df) - len(results_df[results_df['Classified_Category'] == 'Other'])) / len(results_df) * 100:>5.1f}%
Most Common (Other):               {len(results_df[results_df['Classified_Category'] == 'Other']):>6d}        |    % Classified as 'Other':    {len(results_df[results_df['Classified_Category'] == 'Other']) / len(results_df) * 100:>5.1f}%

Model Used:                        {MODEL_NAME}
Device:                            {str(device).upper()}
"""

ax6.text(0.05, 0.95, stats_text, transform=ax6.transAxes, fontsize=11,
         verticalalignment='top', fontfamily='monospace', fontweight='bold',
         bbox=dict(boxstyle='round,pad=1', facecolor='#e8f0f8', edgecolor='#0e4c92', linewidth=2.5))

fig6.patch.set_facecolor('white')

stats_path = os.path.join(RESULTS_DIR, 'statistics_summary.png')
plt.savefig(stats_path, dpi=150, bbox_inches='tight', facecolor='white')
print(f"[OK] Statistics summary saved to: {stats_path}")
plt.show()

print("\n[OK] All visualizations complete!")
print(f"[OK] Saved 6 individual visualizations to: {RESULTS_DIR}")
